# RMIT Hackathon 2025 — Jailbreak Detection

Notebook này tự tìm dữ liệu cuộc thi trong /kaggle/input, chạy baseline TF-IDF và fine-tune Transformer trên GPU. Không cần tải riêng kaggle_train.py.

Trước khi chạy: trong Kaggle chọn Add Input → Competition → RMIT Hackathon 2025, bật GPU và Internet, sau đó chọn Run All. Transformer được bật mặc định; có thể tắt bằng cách đặt RUN_TRANSFORMER = False ở cell cấu hình.

Mục tiêu validation là ROC-AUC. Không có cách bảo đảm đạt 0.999 trước khi đo trên dữ liệu giữ lại và leaderboard. Checkpoint khởi đầu: https://huggingface.co/madhurjindal/Jailbreak-Detector

In [ ]:
from pathlib import Path
import json, random, time, inspect
import numpy as np, pandas as pd, torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import LinearSVC
from scipy.special import expit

SEED = 42
OUT = Path("/kaggle/working")
OUT.mkdir(parents=True, exist_ok=True)
csvs = sorted(Path("/kaggle/input").rglob("*.csv"))
print("CSV files found:", *csvs, sep="\n")

def find_csv(candidates):
    for candidate in candidates:
        matches = [path for path in csvs if path.name.lower() == candidate.lower()]
        if matches:
            return matches[0]
    raise FileNotFoundError(
        f"Could not find {candidates} under /kaggle/input. Add the competition input."
    )

TRAIN_PATH = find_csv(["train.csv", "training.csv"])
TEST_PATH = find_csv(["test.csv"])
SAMPLE_PATH = find_csv(["sample_submission.csv", "sample-submission.csv"])
train_df, test_df, sample_df = [
    pd.read_csv(path) for path in [TRAIN_PATH, TEST_PATH, SAMPLE_PATH]
]
print("train:", train_df.shape, train_df.columns.tolist())
print("test:", test_df.shape, test_df.columns.tolist())
print("sample:", sample_df.shape, sample_df.columns.tolist())
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
TEXT_COL, TARGET_COL = "text", "label"
ID_COL = next((c for c in sample_df.columns if str(c).strip().lower() == "id"), None)
SUB_TARGET_COL = next((c for c in sample_df.columns if str(c).strip().lower() == "target"), None)
if ID_COL is None or SUB_TARGET_COL is None:
    raise ValueError(f"Unexpected sample columns: {sample_df.columns.tolist()}")

for frame, name, required in [
    (train_df, "train", [TEXT_COL, TARGET_COL]),
    (test_df, "test", [TEXT_COL, ID_COL]),
    (sample_df, "sample", [ID_COL, SUB_TARGET_COL]),
]:
    missing = [column for column in required if column not in frame.columns]
    if missing:
        raise ValueError(f"{name} is missing columns: {missing}")
if train_df[TARGET_COL].isna().any():
    raise ValueError("Training labels contain nulls.")
for frame, name in [(test_df, "test"), (sample_df, "sample")]:
    if frame[ID_COL].isna().any() or frame[ID_COL].duplicated().any():
        raise ValueError(f"{name} IDs must be unique and non-null.")

texts = train_df[TEXT_COL].fillna("").astype(str).tolist()
test_texts = test_df[TEXT_COL].fillna("").astype(str).tolist()
raw_labels = train_df[TARGET_COL]
if pd.api.types.is_numeric_dtype(raw_labels):
    if not set(raw_labels.unique()).issubset({0, 1}):
        raise ValueError(f"Expected binary labels, got {raw_labels.unique()[:10]}")
    labels = raw_labels.to_numpy(dtype=np.int64)
else:
    label_map = {"benign": 0, "jailbreak": 1}
    normalized = raw_labels.astype(str).str.strip().str.lower()
    unknown = sorted(set(normalized) - set(label_map))
    if unknown:
        raise ValueError(f"Unknown labels: {unknown}")
    labels = normalized.map(label_map).to_numpy(dtype=np.int64)
if set(np.unique(labels)) != {0, 1}:
    raise ValueError("Training data must contain both classes.")

# Keep duplicate prompts in the same validation fold to avoid leakage.
group_map, groups = {}, []
for index, text in enumerate(texts):
    key = ("missing", index) if not text else ("text", text)
    group_map.setdefault(key, len(group_map))
    groups.append(group_map[key])
groups = np.asarray(groups)
n_splits = min(5, int(np.unique(labels, return_counts=True)[1].min()), len(np.unique(groups)))
if n_splits < 2:
    raise ValueError("Not enough rows/groups for validation.")
folds = []
for train_idx, valid_idx in StratifiedGroupKFold(
    n_splits=n_splits, shuffle=True, random_state=SEED
).split(np.zeros(len(labels)), labels, groups):
    if len(np.unique(labels[valid_idx])) == 2:
        error = abs(len(valid_idx) / len(labels) - 0.20)
        error += abs(float(labels[valid_idx].mean() - labels.mean()))
        folds.append((error, train_idx, valid_idx))
if not folds:
    raise ValueError("Could not create a validation fold containing both labels.")
_, train_idx, valid_idx = min(folds, key=lambda item: item[0])
assert set(groups[train_idx]).isdisjoint(set(groups[valid_idx]))
print("Rows:", len(train_df), len(test_df), len(sample_df))
print("Labels (benign=0, jailbreak=1):", pd.Series(labels).value_counts().sort_index().to_dict())
print("Duplicate train prompts:", int(train_df[TEXT_COL].dropna().duplicated().sum()))
print("Validation split:", len(train_idx), "train /", len(valid_idx), "validation")

In [ ]:
def build_tfidf(mode, c, classifier_name):
    word = TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True)
    char = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=1, sublinear_tf=True)
    features = {
        "word": word,
        "char": char,
        "combined": FeatureUnion([("word", word), ("char", char)]),
    }[mode]
    if classifier_name == "linear_svc":
        classifier = LinearSVC(C=c, class_weight="balanced", max_iter=10000, random_state=SEED)
    else:
        classifier = LogisticRegression(C=c, max_iter=2000, class_weight="balanced")
    return Pipeline([("features", features), ("classifier", classifier)])

def raw_score(model, values):
    if hasattr(model.named_steps["classifier"], "decision_function"):
        return model.decision_function(values)
    return model.predict_proba(values)[:, list(model.classes_).index(1)]

rows = []
for classifier_name in ["linear_svc", "logistic"]:
    for mode in ["word", "char", "combined"]:
        for c in [0.5, 2.0, 4.0]:
            started = time.perf_counter()
            model = build_tfidf(mode, c, classifier_name)
            model.fit([texts[i] for i in train_idx], labels[train_idx])
            auc = roc_auc_score(labels[valid_idx], raw_score(model, [texts[i] for i in valid_idx]))
            rows.append({
                "classifier": classifier_name, "mode": mode, "C": c,
                "roc_auc": float(auc), "seconds": round(time.perf_counter() - started, 2),
            })
experiments = pd.DataFrame(rows).sort_values("roc_auc", ascending=False)
experiments.to_csv(OUT / "experiments.csv", index=False)
display(experiments)
best = experiments.iloc[0]
print("Best TF-IDF validation ROC-AUC:", round(float(best["roc_auc"]), 6))

## Transformer training

Cell kế tiếp chạy tối đa 10 epoch, dừng sớm theo ROC-AUC validation và nạp lại epoch tốt nhất. Cần GPU và Internet để tải checkpoint Hugging Face. Điểm Transformer được đo trên cùng validation split với baseline.

In [ ]:
RUN_TRANSFORMER = True
MODEL_ID = "madhurjindal/Jailbreak-Detector"
MAX_LENGTH = 512
MAX_EPOCHS = 10

if RUN_TRANSFORMER:
    if not torch.cuda.is_available():
        raise RuntimeError("Bật GPU trong Kaggle: Settings → Accelerator → GPU.")
    from scipy.special import softmax
    from transformers import (
        AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding,
        Trainer, TrainingArguments, EarlyStoppingCallback,
    )

    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

    class PromptDataset(torch.utils.data.Dataset):
        def __init__(self, values, targets=None):
            self.encoded = tokenizer(values, truncation=True, max_length=MAX_LENGTH, padding=False)
            self.targets = None if targets is None else np.asarray(targets, dtype=np.int64)
        def __len__(self):
            return len(self.encoded["input_ids"])
        def __getitem__(self, index):
            item = {key: torch.tensor(value[index]) for key, value in self.encoded.items()}
            if self.targets is not None:
                item["labels"] = torch.tensor(self.targets[index], dtype=torch.long)
            return item

    train_ds = PromptDataset([texts[i] for i in train_idx], labels[train_idx])
    valid_ds = PromptDataset([texts[i] for i in valid_idx], labels[valid_idx])
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, num_labels=2)

    def compute_metrics(prediction):
        logits = prediction.predictions[0] if isinstance(prediction.predictions, tuple) else prediction.predictions
        return {"roc_auc": roc_auc_score(prediction.label_ids, softmax(logits, axis=1)[:, 1])}

    strategy_arg = "eval_strategy" if "eval_strategy" in inspect.signature(TrainingArguments).parameters else "evaluation_strategy"
    training_args = TrainingArguments(
        output_dir="/kaggle/working/transformer-checkpoints",
        save_strategy="epoch", logging_strategy="epoch", load_best_model_at_end=True,
        metric_for_best_model="eval_roc_auc", greater_is_better=True, save_total_limit=1,
        num_train_epochs=MAX_EPOCHS, learning_rate=1e-5,
        per_device_train_batch_size=8, per_device_eval_batch_size=16,
        gradient_accumulation_steps=2, weight_decay=0.01, fp16=True,
        seed=SEED, report_to="none", **{strategy_arg: "epoch"},
    )
    trainer = Trainer(
        model=model, args=training_args, train_dataset=train_ds, eval_dataset=valid_ds,
        data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    started = time.perf_counter()
    trainer.train()
    valid_logits = trainer.predict(valid_ds).predictions
    transformer_auc = roc_auc_score(labels[valid_idx], softmax(valid_logits, axis=1)[:, 1])
    print("Transformer validation ROC-AUC:", round(float(transformer_auc), 6))
    print("Best TF-IDF validation ROC-AUC:", round(float(best["roc_auc"]), 6))

    test_ds = PromptDataset(test_texts)
    test_logits = trainer.predict(test_ds).predictions
    test_probability = softmax(test_logits, axis=1)[:, 1]
    if not np.array_equal(test_df[ID_COL].to_numpy(), sample_df[ID_COL].to_numpy()):
        raise ValueError("test.csv IDs/order do not match sample_submission.csv.")
    if len(test_probability) != len(sample_df) or not np.isfinite(test_probability).all():
        raise ValueError("Transformer generated invalid predictions.")

    submission = sample_df.copy()
    submission[SUB_TARGET_COL] = test_probability
    submission.to_csv(OUT / "submission.csv", index=False)
    submission.to_csv(OUT / "submission_transformer.csv", index=False)
    report = {
        "model_id": MODEL_ID, "seed": SEED, "max_epochs": MAX_EPOCHS,
        "transformer_validation_auc": float(transformer_auc),
        "best_tfidf_validation_auc": float(best["roc_auc"]),
        "runtime_seconds": round(time.perf_counter() - started, 2),
        "submission_path": "/kaggle/working/submission.csv",
    }
    (OUT / "transformer_report.json").write_text(json.dumps(report, indent=2))
    print(json.dumps(report, indent=2))
    print("Saved /kaggle/working/submission.csv")
    display(submission.head())
else:
    final_model = build_tfidf(str(best["mode"]), float(best["C"]), str(best["classifier"])).fit(texts, labels)
    scores = raw_score(final_model, test_texts)
    baseline_probability = expit(np.clip(scores, -40, 40))
    if not np.array_equal(test_df[ID_COL].to_numpy(), sample_df[ID_COL].to_numpy()):
        raise ValueError("test.csv IDs/order do not match sample_submission.csv.")
    submission = sample_df.copy()
    submission[SUB_TARGET_COL] = baseline_probability
    submission.to_csv(OUT / "submission.csv", index=False)
    print("Transformer disabled; saved baseline /kaggle/working/submission.csv")
    display(submission.head())

## File được tạo

Sau khi chạy hết notebook, mở phần Output của Kaggle để tải submission.csv. Khi Transformer bật, đây là dự đoán Transformer; submission_transformer.csv là bản sao dự phòng. experiments.csv và transformer_report.json ghi lại kết quả validation.

Điểm validation không phải điểm leaderboard trên tập test ẩn.